# Figure 1 of Zhang et al. (2017), *Understanding deep learning requires rethinking generalization*

This notebook redraws the figure and prints the numbers in the report from the saved logs in `results/`.
Set `RUN_TRAINING = True` to retrain everything with `run_all.sh` first (about 3.5 h on an Apple M4 Pro).

In [ ]:
import json
import subprocess
import sys

import numpy as np
from IPython.display import Image, display

RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(["bash", "run_all.sh"], check=True)


## Parameter counts against Table 1 of the paper

In [ ]:
subprocess.run([sys.executable, "models.py"], check=True)

## Fixed label corruption
The saved logs record the fraction of labels changed. With corruption probability $p$, the expected fraction is $0.9p$. Labels are drawn once and kept fixed across epochs.


In [ ]:
from plot_fig1 import load_all

runs = load_all("results")
for p in (0.2, 0.4, 0.6, 0.8, 1.0):
    run = runs[f"inception_corrupt_p{p:.2f}_s0"]
    print(f"p={p}: changed {run['frac_labels_changed']:.3f}, expected {0.9 * p:.3f}")


## Figure 1
Convergence time is the number of SGD steps until the accuracy on the whole training set first reaches 99%. "Random pixels" permutes all 2352 values of an image; the dashed curve keeps the three color values of a pixel together. Panel (c) uses the real test labels, panel (c') test labels corrupted with the same $p$ as the training labels.

In [ ]:
subprocess.run([sys.executable, "plot_fig1.py"], check=True)
display(Image("figures/fig1_reproduced.png"))

## The paper against my runs
The paper's values were read from the vector graphics of its PDF by `extract_paper_fig1.py`. The last two tables are the diagnostics in the report: the learning curves of panel (a), and the runs with TensorFlow's default initialization.

In [ ]:
subprocess.run([sys.executable, "compare_with_paper.py"], check=True)

## Every run

In [ ]:
from plot_fig1 import load_all, steps_to

print(f"{'run':30s} {'changed':>7s} {'epochs':>6s} {'train acc':>9s} {'test err':>8s} {'best test err':>15s}"
      "   steps to 95 / 99 / 100% train acc")
for tag, run in sorted(load_all("results").items()):
    test_acc = np.array(run["test_acc"])
    changed = run.get("frac_labels_changed")
    changed = "" if changed is None else f"{changed:.3f}"
    best = f"{1 - test_acc.max():.3f} (ep {test_acc.argmax() + 1})"
    steps = " / ".join(str(steps_to(run, t)) for t in (0.95, 0.99, 1.0))
    print(f"{tag:30s} {changed:>7s} {run['epoch'][-1]:6d} {run['train_acc'][-1]:9.4f} "
          f"{1 - test_acc[-1]:8.3f} {best:>15s}   {steps}")

## Panel (b) for other convergence thresholds
Time to overfit relative to $p=0$, for $p = 0, 0.2, \dots, 1$, and the epochs needed at $p=0$ and $p=1$.

In [ ]:
with open("figures/fig1_reproduced_summary.json") as f:
    summary = json.load(f)

for arch, rows in summary.items():
    for th in ("0.95", "0.99", "0.999", "1.0"):
        steps = [r["steps_by_threshold"][th] for r in rows]
        ratios = ["  n/a" if s is None or steps[0] is None else f"{s / steps[0]:5.2f}" for s in steps]
        epochs = [None if s is None else s // 390 for s in (steps[0], steps[-1])]
        print(f"{arch:9s} {th:>5s}: {' '.join(ratios)}   epochs at p=0: {epochs[0]}, at p=1: {epochs[1]}")